In [ ]:
!pip install -U transformers
!pip install langchain==0.1.11 gradio==5.23.2 transformers==4.41.0 bs4==0.0.2 requests==2.31.0 torch==2.2.1
!pip install -U bitsandbytes>=0.46.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 128.5 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.5/807.5 kB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.3/51.3 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 67.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.6/62.6 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.5/755.5 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.2/322.2 kB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━

ERROR: Operation cancelled by user
^C


In [ ]:
#Import Token from Hugging Face for Gemma Model
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
#Import Dependencies for the project
import torch
from transformers import AutoProcessor, PaliGemmaForConditionalGeneration, BitsAndBytesConfig
from PIL import Image
import gradio as gr

from langchain.tools import tool
from langchain.agents import initialize_agent
from langchain.agents import AgentType
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline

In [ ]:
#Create a Local Gemma Model and Run on CUDA/CPU
model_id = "google/paligemma-3b-pt-224"

processor = AutoProcessor.from_pretrained(model_id)

model = PaliGemmaForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

device = "cuda" if torch.cuda.is_available() else "cpu"

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/699 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.26M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

...

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/1.74G [00:00<?, ?B/s]

`config.hidden_act` is ignored, you should use `config.hidden_activation` instead.
Gemma's activation function will be set to `gelu_pytorch_tanh`. Please, use
`config.hidden_activation` if you want to override this behaviour.
See https://github.com/huggingface/transformers/pull/29402 for more details.


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

In [ ]:
#This function:

#Takes an image and converts to Pytorch tensors
#Sends it with a caption prompt to a Gemma model
#The model generates descriptive text
#the text is decoded and cleaned
#Returns the final image caption

def generate_caption(image):

    prompt = "caption\n"

    inputs = processor(
        text=prompt,
        images=image,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=50)

    caption = processor.decode(output[0], skip_special_tokens=True)

    if caption.startswith(prompt):
        caption = caption[len(prompt):]

    return caption.strip()

In [ ]:
#Function used to define Tool to be passed to LangChain
@tool
def image_caption_tool(image_path: str) -> str:
    """
    Generates caption for an input image using PaliGemma.
    """

    image = Image.open(image_path).convert("RGB")

    caption = generate_caption(image)

    return caption

In [ ]:
#Create a HuggingFace Pipeline using Gemma-2b-it[Text Generation Model] for Image Summary
from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline

text_pipe = pipeline(
    "text-generation",
    model="google/gemma-2b-it",
    max_new_tokens=80,
    repetition_penalty=2.0,
    no_repeat_ngram_size=3,
    return_full_text=False,
    device_map="auto"
)

llm = HuggingFacePipeline(pipeline=text_pipe)

config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/67.1M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

In [ ]:
#Use the LLM created Above to create a LLMChain [LangChain API]
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain

summary_prompt = PromptTemplate(
    input_variables=["caption"],
    template="<start_of_turn>user\nSummarize this image caption in one or two sentences.\nCaption: {caption}<end_of_turn>\n<start_of_turn>model\n"
)

summary_chain = LLMChain(
    llm=llm,
    prompt=summary_prompt
)


In [ ]:
#Create an LangChain LLM Agent
tools = [image_caption_tool]

agent = initialize_agent(
    tools,
    llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

/usr/local/lib/python3.12/dist-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: The function `initialize_agent` was deprecated in LangChain 0.1.0 and will be removed in 0.2.0. Use Use new agent constructor methods like create_react_agent, create_json_agent, create_structured_chat_agent, etc. instead.
  warn_deprecated(


In [ ]:
def clean_summary(text):
    """Remove repeated/degenerate words and extract clean summary."""
    # Extract text after "Summary:" if present
    if "Summary:" in text:
        text = text.split("Summary:")[-1].strip()

    # Remove word-level repetition (e.g., "Wildlife Wildlife Wildlife")
    words = text.split()
    cleaned = []
    for w in words:
        if not cleaned or w.lower() != cleaned[-1].lower():
            cleaned.append(w)
    text = " ".join(cleaned)

    # Keep only first 2 sentences
    parts = text.split(".")
    sentences = [s.strip() for s in parts if s.strip()][:2]
    return ". ".join(sentences) + "." if sentences else text

In [ ]:
#Create a Main Function for Gradio
def analyze_image(image):

    image_path = "temp_image.jpg"
    image.save(image_path)

    caption = image_caption_tool.run(image_path)

    raw_summary = summary_chain.run(caption=caption)

    summary = clean_summary(raw_summary)

    result = f"Caption: {caption}\n\nSummary: {summary}"

    return result

In [ ]:
#Create the Gradio Interface
interface = gr.Interface(
    fn=analyze_image,
    inputs=gr.Image(type="pil"),
    outputs="text",
    title="LangChain Image Caption Agent (PaliGemma)",
    description="Upload an image. The agent generates a caption and summary."
)

interface.launch()

Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://99a2f455312aa76b76.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
